# Experiment 6 — PyDI Knowledge Base Fusion Baseline

This notebook implements a **KB-only baseline** using PyDI's normalization and data fusion modules.
No LLM, no RAG retrieval — instead we directly fuse values from KB rows that share the same
`cluster_id` as the query product.

## PyDI Modules Used
| Module | Class / Function | Purpose |
|--------|------------------|---------|
| `PyDI.normalization` | `NormalizationSpec`, `transform_dataframe` | Clean KB values before fusion |
| `PyDI.fusion` | `DataFusionStrategy` | Define per-attribute conflict resolution |
| `PyDI.fusion` | `DataFusionEngine` | Execute fusion across KB datasets |
| `PyDI.fusion` | `DataFusionEvaluator` | Evaluate fused results vs ground truth |
| `PyDI.fusion` | `voting`, `median` | Conflict resolvers (text / numeric) |
| `PyDI.fusion` | `tokenized_match`, `numeric_tolerance_match` | Evaluation functions |

## Experimental Design
- **What it answers**: What is the upper bound of a KB-only approach — i.e., if we skip the LLM
  entirely and just look up + fuse the KB?
- **Conflict resolution**: `voting` for text attributes (most frequent value wins across DS2/3/4),
  `median` for numeric attributes (robust to outliers).
- **Evaluation**: Same functions as exp_runner (`is_correct_standard`, `evaluate_ce`) for
  apples-to-apples comparison, PLUS PyDI's `DataFusionEvaluator` for framework-native metrics.
- **Expected outcome**: High recall on common attributes (bus_type, numeric), lower on
  model_number — because fusion cannot resolve near-identical SKUs without LLM reasoning.

## Prerequisites
Run `exp_setup.ipynb` first. This notebook expects:
- `normalized_products/dataset_[1-4]_normalized.json`
- `eval_set.csv`, `query_indices.csv`
- `results/exp1_llm_only.csv` (for comparison table)

## 1. Environment Setup

In [48]:
import sys
sys.path.insert(0, '/home/ma/ma_ma/ma_mpandya/RAG_Data_Cleaning/PyDI/venv/lib64/python3.12/site-packages')
sys.path.insert(0, '/home/ma/ma_ma/ma_mpandya/RAG_Data_Cleaning/PyDI/venv/lib/python3.12/site-packages')
sys.path.append('/home/ma/ma_ma/ma_mpandya/RAG_Data_Cleaning/PyDI')

import os, re, math
import numpy as np
import pandas as pd
from sentence_transformers import CrossEncoder

# ── PyDI imports ──────────────────────────────────────────────────────────────
from PyDI.normalization import NormalizationSpec, transform_dataframe
from PyDI.fusion import (
    DataFusionStrategy,
    DataFusionEngine,
    DataFusionEvaluator,
    voting,
    median,
    longest_string,
    tokenized_match,
    numeric_tolerance_match,
)

print('PyDI imports OK')

PyDI imports OK


## 2. Configuration
Mirrors exp_runner.ipynb exactly.

In [49]:
TARGET_ATTRIBUTES  = ['bus_type', 'model_number', 'model',
                      'read_speed_mb_s', 'write_speed_mb_s', 'height_mm', 'width_mm']
NUMERIC_ATTRIBUTES = {'read_speed_mb_s', 'write_speed_mb_s', 'height_mm', 'width_mm'}
TEXT_ATTRIBUTES    = {'bus_type', 'model_number', 'model'}

RESULTS_DIR = 'results'
EXP6_FILE   = 'exp6_pydi_fusion.csv'
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f'Results will be saved to: {RESULTS_DIR}/{EXP6_FILE}')

Results will be saved to: results/exp6_pydi_fusion.csv


## 3. Load Datasets and Eval Set

In [50]:
DATA_DIR = 'normalized_products'
df1 = pd.read_json(f'{DATA_DIR}/dataset_1_normalized.json')
df2 = pd.read_json(f'{DATA_DIR}/dataset_2_normalized.json')
df3 = pd.read_json(f'{DATA_DIR}/dataset_3_normalized.json')
df4 = pd.read_json(f'{DATA_DIR}/dataset_4_normalized.json')

assert os.path.exists('eval_set.csv'),      'Run exp_setup.ipynb first'
assert os.path.exists('query_indices.csv'), 'Run exp_setup.ipynb first'

eval_df       = pd.read_csv('eval_set.csv')
query_indices = pd.read_csv('query_indices.csv').iloc[:, 0].tolist()
query_df      = df1.loc[query_indices].copy()

print(f'Eval tasks: {len(eval_df)} | Query rows: {len(query_df)}')
print('Tasks per attribute:')
print(eval_df['attribute'].value_counts().to_string())

Eval tasks: 96 | Query rows: 50
Tasks per attribute:
attribute
model_number        23
read_speed_mb_s     15
bus_type            13
height_mm           13
width_mm            12
write_speed_mb_s    10
model               10


## 4. PyDI Step 1 — Normalize KB Datasets

Before fusion we normalize the KB using `PyDI.normalization`:
- Strip whitespace from all string columns
- Cast numeric target attributes to `float`
- Lowercase text target attributes for consistent voting

This addresses a real problem: the same bus_type may appear as `'PCIe 3.0 x4 '` (trailing space)
in one dataset and `'PCIe 3.0 x4'` in another — normalization ensures `voting` counts these as
the same value.

In [51]:
def normalize_kb_dataset(df, dataset_name):
    """Apply PyDI normalization to a KB dataset."""
    df = df.copy()
    df.attrs['dataset_name'] = dataset_name

    spec = NormalizationSpec()

    # Text attributes: strip whitespace + lowercase for consistent voting
    for attr in TEXT_ATTRIBUTES:
        if attr in df.columns:
            spec.set_column(attr,
                            output_type='string',
                            strip_whitespace=True,
                            case='lower',
                            on_failure='keep')

    # Numeric attributes: cast to float
    for attr in NUMERIC_ATTRIBUTES:
        if attr in df.columns:
            spec.set_column(attr,
                            output_type='float',
                            strip_whitespace=True,
                            on_failure='null')

    result = transform_dataframe(df, spec)
    normalized = result.dataframe
    normalized.attrs['dataset_name'] = dataset_name

    print(f'  {dataset_name}: {result.total_transformed} values normalized, '
          f'{result.total_failed} failures')
    return normalized


print('Normalizing KB datasets with PyDI...')
df2_norm = normalize_kb_dataset(df2, 'dataset_2')
df3_norm = normalize_kb_dataset(df3, 'dataset_3')
df4_norm = normalize_kb_dataset(df4, 'dataset_4')

kb_norm = pd.concat([df2_norm, df3_norm, df4_norm], ignore_index=True)
print(f'\nNormalized KB: {len(kb_norm):,} rows')

# Quick sanity check — show before/after for bus_type
raw_sample   = pd.concat([df2, df3, df4], ignore_index=True)
print('\nSample bus_type values (raw vs normalized):')
for raw, norm in zip(
    raw_sample['bus_type'].dropna().unique()[:5],
    kb_norm['bus_type'].dropna().unique()[:5]
):
    print(f'  {repr(raw):30s} → {repr(norm)}')

Normalizing KB datasets with PyDI...
  dataset_2: 1658 values normalized, 0 failures
  dataset_3: 1551 values normalized, 0 failures
  dataset_4: 1306 values normalized, 0 failures

Normalized KB: 2,200 rows

Sample bus_type values (raw vs normalized):
  'SATA III'                     → 'sata iii'
  'PCIe 3.0 x4'                  → 'pcie 3.0 x4'
  'USB 3.0'                      → 'usb 3.0'
  'PCIe 3.0 x16'                 → 'pcie 3.0 x16'
  'SAS 3G'                       → 'sas 3g'


## 5. PyDI Step 2 — Define Fusion Strategy

We define attribute-level conflict resolution using `DataFusionStrategy`:
- **Text attributes** → `voting`: pick the most frequent value across DS2, DS3, DS4
- **Numeric attributes** → `median`: robust to outlier values in individual datasets
- **model_number** → `longest_string`: model numbers shouldn't be averaged; longest is most
  specific and least likely to be a truncated variant

We also add PyDI evaluation functions that mirror our existing eval logic.

In [52]:
strategy = DataFusionStrategy('rag_kb_fusion_strategy')

# ── Conflict resolution functions ─────────────────────────────────────────────
# bus_type: voting — most common value across datasets wins
strategy.add_attribute_fuser('bus_type',         voting)
# model: voting
strategy.add_attribute_fuser('model',            voting)
# model_number: longest_string — most specific / least truncated
strategy.add_attribute_fuser('model_number',     longest_string)
# Numeric: median — robust to single-dataset outliers
strategy.add_attribute_fuser('read_speed_mb_s',  median)
strategy.add_attribute_fuser('write_speed_mb_s', median)
strategy.add_attribute_fuser('height_mm',        median)
strategy.add_attribute_fuser('width_mm',         median)

# ── Evaluation functions (used by DataFusionEvaluator) ────────────────────────
# Text: tokenized Jaccard similarity (threshold=0.8 mirrors our CE eval leniency)
strategy.add_evaluation_function('bus_type',         tokenized_match, threshold=0.8)
strategy.add_evaluation_function('model',            tokenized_match, threshold=0.8)
strategy.add_evaluation_function('model_number',     tokenized_match, threshold=0.9)
# Numeric: 10% tolerance (matches is_correct_standard)
strategy.add_evaluation_function('read_speed_mb_s',  numeric_tolerance_match, tolerance=0.10)
strategy.add_evaluation_function('write_speed_mb_s', numeric_tolerance_match, tolerance=0.10)
strategy.add_evaluation_function('height_mm',        numeric_tolerance_match, tolerance=0.10)
strategy.add_evaluation_function('width_mm',         numeric_tolerance_match, tolerance=0.10)

print('Fusion strategy configured:')
for attr in TARGET_ATTRIBUTES:
    resolver = 'voting' if attr in ('bus_type', 'model') else \
               'longest_string' if attr == 'model_number' else 'median'
    eval_fn  = 'tokenized_match' if attr in TEXT_ATTRIBUTES else 'numeric_tolerance_match(0.10)'
    print(f'  {attr:<22} resolver={resolver:<15} eval={eval_fn}')

Fusion strategy configured:
  bus_type               resolver=voting          eval=tokenized_match
  model_number           resolver=longest_string  eval=tokenized_match
  model                  resolver=voting          eval=tokenized_match
  read_speed_mb_s        resolver=median          eval=numeric_tolerance_match(0.10)
  write_speed_mb_s       resolver=median          eval=numeric_tolerance_match(0.10)
  height_mm              resolver=median          eval=numeric_tolerance_match(0.10)
  width_mm               resolver=median          eval=numeric_tolerance_match(0.10)


## 6. PyDI Step 3 — Build Correspondences and Run Fusion

PyDI's `DataFusionEngine` requires a correspondences DataFrame with `id1`/`id2` columns
linking records that describe the same entity. We derive these directly from `cluster_id` —
all KB rows sharing the same `cluster_id` refer to the same real-world product.

In [53]:
# Add stable integer _id column required by DataFusionEngine
df2_norm = df2_norm.reset_index(drop=True).copy()
df3_norm = df3_norm.reset_index(drop=True).copy()
df4_norm = df4_norm.reset_index(drop=True).copy()

df2_norm['_id'] = df2_norm.index + 10000
df3_norm['_id'] = df3_norm.index + 20000
df4_norm['_id'] = df4_norm.index + 30000

df2_norm.attrs['dataset_name'] = 'dataset_2'
df3_norm.attrs['dataset_name'] = 'dataset_3'
df4_norm.attrs['dataset_name'] = 'dataset_4'

# Build all KB into one frame for fast cluster_id lookup
kb_all = pd.concat([df2_norm, df3_norm, df4_norm], ignore_index=True)

# Build correspondences: pairs of _id that share the same cluster_id
# We only need correspondences for cluster_ids that appear in our eval set
eval_cluster_ids = set()
for _, task in eval_df.iterrows():
    idx = task['df1_idx']
    eval_cluster_ids.add(query_df.loc[idx, 'cluster_id'])

print(f'Eval cluster IDs to fuse: {len(eval_cluster_ids)}')

corr_rows = []
for cid in eval_cluster_ids:
    matching = kb_all[kb_all['cluster_id'] == cid]['_id'].tolist()
    # Create all pairs within this cluster
    for i in range(len(matching)):
        for j in range(i + 1, len(matching)):
            corr_rows.append({'id1': matching[i], 'id2': matching[j]})
        # Also pair with itself so singletons are included
        if len(matching) == 1:
            corr_rows.append({'id1': matching[i], 'id2': matching[i]})

correspondences = pd.DataFrame(corr_rows)
print(f'Correspondence pairs: {len(correspondences):,}')
print(f'KB rows involved: {kb_all[kb_all["cluster_id"].isin(eval_cluster_ids)].shape[0]}')

Eval cluster IDs to fuse: 50
Correspondence pairs: 130
KB rows involved: 140


In [54]:
corr_rows = []
for cid in eval_cluster_ids:
    matching = kb_all[kb_all['cluster_id'] == cid]['_id'].tolist()
    for i in range(len(matching)):
        for j in range(i + 1, len(matching)):
            corr_rows.append({'id1': matching[i], 'id2': matching[j], 'score': 1.0})
        if len(matching) == 1:
            corr_rows.append({'id1': matching[i], 'id2': matching[i], 'score': 1.0})

correspondences = pd.DataFrame(corr_rows)
print(f'Correspondence pairs: {len(correspondences):,}')

Correspondence pairs: 130


In [55]:
# Run PyDI DataFusionEngine
engine = DataFusionEngine(
    strategy,
    debug=True,
    debug_file=f'{RESULTS_DIR}/exp6_fusion_debug.jsonl',
    debug_format='json',
)

print('Running PyDI DataFusionEngine...')
fused_df = engine.run(
    datasets=[df2_norm, df3_norm, df4_norm],
    correspondences=correspondences,
    id_column='_id',
    include_singletons=True,   # keep clusters that only appear in one dataset
)

print(f'Fused records: {len(fused_df)}')
print(f'Columns: {list(fused_df.columns[:10])} ...')
print('\nSample fused record:')
sample = fused_df[fused_df['cluster_id'].isin(list(eval_cluster_ids)[:1])]
if len(sample):
    print(sample[['cluster_id'] + TARGET_ATTRIBUTES].head(1).to_string())

Running PyDI DataFusionEngine...
Fused records: 2110
Columns: ['_id', '_fusion_sources', '_fusion_source_datasets', 'brand', 'bus_type', 'chipset_name', 'cluster_id', 'color', 'description', 'form_factor'] ...

Sample fused record:
    cluster_id bus_type     model_number    model  read_speed_mb_s  write_speed_mb_s  height_mm  width_mm
48     1882499      sas  st3600057ss-eql  cheetah              NaN               NaN        NaN       NaN


## 7. Evaluation Functions
Identical to exp_runner.ipynb — copied verbatim for apples-to-apples comparison.

In [56]:
import glob
HF_CACHE = '/home/ma/ma_ma/ma_mpandya/.cache/huggingface/hub'
CE_SNAP  = glob.glob(f'{HF_CACHE}/models--cross-encoder--ms-marco-MiniLM-L-6-v2/snapshots/*/')
CE_PATH  = CE_SNAP[0].rstrip('/') if CE_SNAP else 'cross-encoder/ms-marco-MiniLM-L-6-v2'
cross_encoder = CrossEncoder(CE_PATH)
print(f'CrossEncoder loaded from: {CE_PATH}')


def is_correct_standard(predicted, ground_truth, attribute):
    if not predicted or str(predicted).strip().lower() in {'', 'nan', 'none', 'unknown', 'null'}:
        return False
    if attribute in NUMERIC_ATTRIBUTES:
        try:
            p = float(str(predicted).replace(',', '').strip())
            g = float(str(ground_truth).replace(',', '').strip())
            return abs(p - g) / abs(g) <= 0.10 if g != 0 else p == 0
        except:
            pass
    p, g = str(predicted).lower().strip(), str(ground_truth).lower().strip()
    return p == g or p in g or g in p


def evaluate_ce(predicted, ground_truth, attribute):
    if predicted == 'UNKNOWN' or str(predicted).lower() in {'nan', 'none', 'null', ''}:
        return 'wrong'
    if attribute in NUMERIC_ATTRIBUTES:
        try:
            p = float(str(predicted).replace(',', '').strip())
            g = float(str(ground_truth).replace(',', '').strip())
            if g == 0: return 'correct' if p == 0 else 'wrong'
            r = abs(p - g) / abs(g)
            return 'correct' if r <= 0.10 else ('acceptable' if r <= 0.30 else 'wrong')
        except:
            return 'wrong'
    score = cross_encoder.predict([[ground_truth, predicted]])[0]
    return 'correct' if score > 2.0 else ('acceptable' if score > -1.0 else 'wrong')


def fix_prediction(pred):
    if isinstance(pred, str) and pred.strip().upper().startswith('VALUE:'):
        val = pred.strip().split(':', 1)[1].strip()
        return 'UNKNOWN' if val.upper() in {'UNKNOWN', 'NONE', 'NULL', 'NAN', ''} else val
    return pred


def evaluate_and_save(results_df, config_name, filename):
    results_df['predicted']        = results_df['predicted'].apply(fix_prediction)
    results_df['unknown']          = results_df['predicted'] == 'UNKNOWN'
    results_df['correct_standard'] = results_df.apply(
        lambda r: is_correct_standard(r['predicted'], r['ground_truth'], r['attribute']), axis=1)
    results_df['ce_judgment'] = [
        evaluate_ce(r['predicted'], r['ground_truth'], r['attribute'])
        for _, r in results_df.iterrows()]
    path = f'{RESULTS_DIR}/{filename}'
    results_df.to_csv(path, index=False)

    std = results_df['correct_standard'].mean()
    ce  = results_df['ce_judgment'].isin(['correct', 'acceptable']).mean()
    unk = results_df['unknown'].mean()
    print(f'\n{"="*60}\nRESULTS — {config_name}\n{"="*60}')
    print(f'Standard accuracy:    {std:.3f} ({std*100:.1f}%)')
    print(f'CE eval (c+a):        {ce:.3f} ({ce*100:.1f}%)')
    print(f'UNKNOWN rate:         {unk:.3f} ({unk*100:.1f}%)')
    print(f'Total tasks:          {len(results_df)}')
    print('\nPer-attribute:')
    print(results_df.groupby('attribute').agg(
        n=('correct_standard', 'count'),
        std_acc=('correct_standard', 'mean'),
        ce_acc=('ce_judgment', lambda x: x.isin(['correct', 'acceptable']).mean()),
        unknown=('unknown', 'mean')
    ).round(3).to_string())
    print(f'\n✓ Saved to {path}')
    return results_df

print('Evaluation functions ready.')

CrossEncoder loaded from: /home/ma/ma_ma/ma_mpandya/.cache/huggingface/hub/models--cross-encoder--ms-marco-MiniLM-L-6-v2/snapshots/c5ee24cb16019beea0893ab7796b1df96625c6b8
Evaluation functions ready.


## 8. Experiment 6 — PyDI KB Fusion Predictions

For each eval task:
1. Look up the query's `cluster_id` in `fused_df` (the PyDI-fused KB)
2. Read the fused value for the target attribute
3. If no fused record exists → predict `UNKNOWN`
4. Evaluate with the same functions as all other experiments

In [57]:
# Build a lookup: cluster_id → fused row (one row per cluster after fusion)
# fused_df may have multiple rows per cluster_id if engine doesn't collapse — group and take first
fused_lookup = (
    fused_df
    .groupby('cluster_id')
    .first()
    .reset_index()
)
fused_lookup = fused_lookup.set_index('cluster_id')
print(f'Fused lookup: {len(fused_lookup)} unique cluster_ids')
print(f'Eval cluster_ids covered: '
      f'{sum(1 for cid in eval_cluster_ids if cid in fused_lookup.index)} / {len(eval_cluster_ids)}')

Fused lookup: 812 unique cluster_ids
Eval cluster_ids covered: 50 / 50


In [58]:
EXP6_FULL_PATH = f'{RESULTS_DIR}/{EXP6_FILE}'

if os.path.exists(EXP6_FULL_PATH):
    print('Loading existing Exp 6 results...')
    exp6_df = pd.read_csv(EXP6_FULL_PATH)
else:
    print('Running Exp 6 — PyDI KB Fusion...')
    predictions = []

    for i, (_, task) in enumerate(eval_df.iterrows()):
        idx        = task['df1_idx']
        attr       = task['attribute']
        gt         = task['ground_truth']
        cluster_id = query_df.loc[idx, 'cluster_id']

        # Look up fused value for this cluster_id
        predicted = 'UNKNOWN'
        if cluster_id in fused_lookup.index:
            fused_row = fused_lookup.loc[cluster_id]
            raw_val   = fused_row.get(attr, None)
            if raw_val is not None and pd.notna(raw_val):
                val_str = str(raw_val).strip()
                if val_str.lower() not in {'', 'nan', 'none', 'null'}:
                    predicted = val_str

        print(f'  [{i+1}/{len(eval_df)}] cluster={cluster_id} | {attr:<22} | '
              f'GT: {str(gt):<25} | Fused: {predicted}')

        predictions.append({
            'df1_idx':    idx,
            'config':     'PyDI-KB-Fusion',
            'attribute':  attr,
            'is_numeric': task['is_numeric'],
            'ground_truth': gt,
            'predicted':  predicted,
            'unknown':    predicted == 'UNKNOWN',
            'cluster_id': cluster_id,
        })

    exp6_df = evaluate_and_save(pd.DataFrame(predictions), 'Exp 6: PyDI KB Fusion', EXP6_FILE)

Loading existing Exp 6 results...


## 9. PyDI Step 4 — DataFusionEvaluator

Run PyDI's native evaluator against the ground truth. This uses the evaluation functions
registered in the strategy (`tokenized_match`, `numeric_tolerance_match`) and produces
framework-native accuracy metrics alongside our custom ones.

In [59]:
# Build a ground truth DataFrame from eval_set in the format PyDI expects
# One row per cluster_id, one column per attribute
gt_pivot = (
    eval_df
    .assign(cluster_id=eval_df['df1_idx'].map(lambda idx: query_df.loc[idx, 'cluster_id']))
    .pivot_table(index='cluster_id', columns='attribute',
                 values='ground_truth', aggfunc='first')
    .reset_index()
)
gt_pivot['_id'] = range(len(gt_pivot))

# Align fused_df to same cluster_ids
fused_for_eval = fused_lookup.reset_index()
fused_for_eval = fused_for_eval[fused_for_eval['cluster_id'].isin(gt_pivot['cluster_id'])].copy()
fused_for_eval['_id'] = fused_for_eval['cluster_id'].map(
    dict(zip(gt_pivot['cluster_id'], gt_pivot['_id']))
)

print(f'GT rows: {len(gt_pivot)} | Fused rows for eval: {len(fused_for_eval)}')

try:
    evaluator = DataFusionEvaluator(
        strategy,
        debug=True,
        fusion_debug_logs=f'{RESULTS_DIR}/exp6_fusion_debug.jsonl'
    )
    metrics = evaluator.evaluate(
        fused_df=fused_for_eval,
        fused_id_column='_id',
        expected_df=gt_pivot,
        expected_id_column='_id'
    )
    print('\nPyDI DataFusionEvaluator metrics:')
    for k, v in metrics.items():
        if isinstance(v, (int, float)):
            print(f'  {k}: {round(v, 3)}')
except Exception as e:
    print(f'DataFusionEvaluator note: {e}')
    print('(Falling back to custom evaluation above — results already computed)')

GT rows: 50 | Fused rows for eval: 50

PyDI DataFusionEvaluator metrics:
  overall_accuracy: 0.993
  macro_accuracy: 0.995
  num_evaluated_records: 50
  num_evaluated_attributes: 8
  total_evaluations: 146
  total_correct: 145
  read_speed_mb_s_accuracy: 1.0
  read_speed_mb_s_count: 15
  cluster_id_accuracy: 1.0
  cluster_id_count: 50
  bus_type_accuracy: 1.0
  bus_type_count: 13
  model_number_accuracy: 0.957
  model_number_count: 23
  write_speed_mb_s_accuracy: 1.0
  write_speed_mb_s_count: 10
  height_mm_accuracy: 1.0
  height_mm_count: 13
  model_accuracy: 1.0
  model_count: 10
  width_mm_accuracy: 1.0
  width_mm_count: 12


## 10. Full Comparison — All Experiments
Loads all experiment results including Exp 6 and prints a unified table.

In [62]:
all_files = {
    'Exp1: LLM-only':        'results_mit_UNKNOWN/exp1_llm_only.csv',
    'Exp2: RAG-MiniLM':      'results_mit_UNKNOWN/exp2_rag_minilm.csv',
    'Exp3: MiniLM+Reranker': 'results_mit_UNKNOWN/exp3_rag_minilm_reranker.csv',
    'Exp4: BGE+Reranker':    'results_mit_UNKNOWN/exp4_rag_bge_reranker.csv',
    'Exp5: OpenAI+Reranker': 'results_mit_UNKNOWN/exp5_rag_openai_reranker.csv',
    'Exp6: PyDI-KB-Fusion':  'results/exp6_pydi_fusion.csv',
}

print(f'{"Configuration":<35} {"Std acc":>10} {"CE eval":>10} {"UNKNOWN":>10} {"n":>6}')
print('-' * 75)

all_dfs = {}
for name, path in all_files.items():
    if not os.path.exists(path):
        print(f'  {name:<35} (not run yet)')
        continue
    df = pd.read_csv(path)
    all_dfs[name] = df
    std = df['correct_standard'].mean()
    ce  = df['ce_judgment'].isin(['correct', 'acceptable']).mean()
    unk = df['unknown'].mean()
    marker = ' ← PyDI fusion' if 'Fusion' in name else ''
    print(f'  {name:<35} {std:>10.3f} {ce:>10.3f} {unk:>10.3f} {len(df):>6}{marker}')

Configuration                          Std acc    CE eval    UNKNOWN      n
---------------------------------------------------------------------------
  Exp1: LLM-only                           0.177      0.219      0.448     96
  Exp2: RAG-MiniLM                         0.417      0.448      0.177     96
  Exp3: MiniLM+Reranker                    0.740      0.760      0.083     96
  Exp4: BGE+Reranker                       0.760      0.781      0.031     96
  Exp5: OpenAI+Reranker                    0.771      0.833      0.021     96
  Exp6: PyDI-KB-Fusion                     0.990      1.000      0.000     96 ← PyDI fusion


In [63]:
# Per-attribute breakdown including Exp 6
if len(all_dfs) > 1:
    print('\nPer-attribute standard accuracy (all experiments):')
    combined = pd.concat(all_dfs.values(), ignore_index=True)
    pivot = (
        combined
        .groupby(['config', 'attribute'])['correct_standard']
        .mean()
        .unstack('config')
        .round(3)
    )
    print(pivot.to_string())


Per-attribute standard accuracy (all experiments):
config            LLM-only  PyDI-KB-Fusion  RAG-BGE-Reranker  RAG-MiniLM  RAG-MiniLM-Reranker  RAG-OpenAI-Reranker
attribute                                                                                                         
bus_type             0.538           1.000             0.923       0.846                0.769                0.846
height_mm            0.000           1.000             0.769       0.154                0.769                0.846
model                0.100           1.000             0.500       0.400                0.500                0.400
model_number         0.043           0.957             0.696       0.217                0.652                0.696
read_speed_mb_s      0.200           1.000             0.800       0.533                0.933                1.000
width_mm             0.083           1.000             0.750       0.417                0.583                0.750
write_speed_mb_s     0.400  

## 11. Interpretation Guide

Use these results to answer the question:
**"Does the LLM add value on top of pure KB fusion?"**

| Exp6 vs others | Interpretation |
|----------------|----------------|
| Exp6 >> LLM-only | KB fusion alone beats LLM-only — retrieval precision matters more than LLM reasoning |
| Exp6 ~ RAG-MiniLM | Direct fusion as good as retrieval+LLM — simpler approach competitive |
| Exp6 << TE+RR | Best RAG config adds significant value over raw fusion — LLM reasoning helps |
| Exp6 high on numeric, low on model_number | Confirms LLM needed for fine-grained disambiguation |

**Key insight for presentation**: Exp6 is the oracle upper bound for KB-only methods.
Any RAG config that exceeds Exp6 demonstrates that LLM extraction adds genuine value
beyond what the KB structure alone provides.